# Email Spam Detection Using Logistic Regression

## Project Overview
This project builds an end-to-end binary classification system for detecting whether an email is **Routine/Not Spam** or **Spam** using measurable email-related features.

> **Important dataset alignment:** The supplied dataset is named `spam_detection_dataset.csv` and its target column is `is_spam`. It contains 20,000 rows and six columns: `num_links`, `num_words`, `has_offer`, `sender_score`, `all_caps`, and `is_spam`. Therefore, this implementation correctly treats the task as **email spam detection**, not threat detection. If your official college brief specifically requires "Potentially Threatening" labels, a dataset with that target must be used instead of silently relabeling spam as threat.

### Objectives
1. Inspect and clean the dataset.
2. Analyze missing values, duplicates, and class balance.
3. Perform exploratory data analysis.
4. Prepare features and target.
5. Split the data into training and testing sets.
6. Train a Logistic Regression classifier.
7. Evaluate it using Accuracy, Precision, Recall, F1-score, ROC-AUC, and a Confusion Matrix.
8. Save the trained model for deployment.
9. Provide a simple Streamlit prototype for new predictions.

### Model
**Logistic Regression** is used as the main model because it is appropriate for binary classification, fast to train, interpretable, and easy to explain in a viva.


In [1]:
# 1. Import required libraries

import os
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
DATA_PATH = "spam_detection_dataset.csv"
MODEL_PATH = "email_spam_model.pkl"

print("Libraries imported successfully.")


Libraries imported successfully.


In [19]:
import os
print(os.getcwd())

/content


In [21]:
# 2. Load dataset

import os
import pandas as pd

DATA_PATH = "spam_detection_dataset.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Place spam_detection_dataset.csv in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head(10))

FileNotFoundError: Dataset not found: spam_detection_dataset.csv
Place spam_detection_dataset.csv in the same folder as this notebook.

In [12]:
# 3. Dataset structure and data types

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()


Column names:


NameError: name 'df' is not defined

In [ ]:
# 4. Data quality checks

print("Missing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("Duplicate rows:", df.duplicated().sum())

print("\nDescriptive statistics:")
display(df.describe(include="all"))


In [ ]:
# 5. Clean the dataset

# Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"Removed {before - after} duplicate rows.")

# Handle missing values in numeric columns using median
numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("\nMissing values after cleaning:")
display(df.isnull().sum().to_frame("missing_values"))
print("\nCleaned dataset shape:", df.shape)


In [ ]:
# 6. Target distribution / class balance

target = "is_spam"

if target not in df.columns:
    raise ValueError("Expected target column 'is_spam' was not found.")

class_counts = df[target].value_counts().sort_index()
class_percent = df[target].value_counts(normalize=True).sort_index().mul(100)

print("Class counts:")
display(class_counts.to_frame("count"))

print("Class percentages:")
display(class_percent.round(2).to_frame("percentage"))

plt.figure(figsize=(7, 5))
sns.countplot(data=df, x=target)
plt.title("Email Spam Class Distribution")
plt.xlabel("Class (0 = Not Spam, 1 = Spam)")
plt.ylabel("Number of Emails")
plt.show()


## 7. Exploratory Data Analysis

The dataset contains five predictor variables:
- `num_links`: number of links in the email
- `num_words`: number of words
- `has_offer`: indicator for offer-related content
- `sender_score`: numerical sender-related score
- `all_caps`: indicator for excessive/all-capital text

The target is:
- `is_spam = 0`: Not Spam
- `is_spam = 1`: Spam


In [ ]:
# 8. Distribution of numerical features

feature_columns = [
    "num_links",
    "num_words",
    "has_offer",
    "sender_score",
    "all_caps"
]

missing_features = [c for c in feature_columns if c not in df.columns]
if missing_features:
    raise ValueError(f"Missing expected feature columns: {missing_features}")

for col in feature_columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x=col, hue=target, kde=True, element="step")
    plt.title(f"Distribution of {col} by Spam Class")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


In [ ]:
# 9. Feature correlation matrix

plt.figure(figsize=(8, 6))
sns.heatmap(
    df[feature_columns + [target]].corr(),
    annot=True,
    fmt=".2f",
    cmap="Blues"
)
plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
# 10. Compare feature averages by class

class_means = df.groupby(target)[feature_columns].mean().T
display(class_means)

class_means.plot(kind="bar", figsize=(10, 5))
plt.title("Average Feature Values by Email Class")
plt.xlabel("Feature")
plt.ylabel("Average Value")
plt.xticks(rotation=45)
plt.legend(title="is_spam")
plt.tight_layout()
plt.show()


## 11. Feature Preparation

Only the five predictor columns are used for modelling. The target column `is_spam` is excluded from the feature matrix.

A `Pipeline` combines standardization and Logistic Regression. Standardization is important because the features are on different numerical scales.

The classifier uses `class_weight="balanced"` because the dataset is imbalanced. This prevents the model from simply favoring the majority class.


In [ ]:
# 12. Prepare X and y

X = df[feature_columns].copy()
y = df[target].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


In [ ]:
# 13. Train Logistic Regression model

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

print("Model training completed.")


In [ ]:
# 14. Predictions

y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

print("First 10 predictions:", y_pred[:10])
print("First 10 spam probabilities:", np.round(y_probability[:10], 3))


In [ ]:
# 15. Evaluation metrics

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

display(results.style.format({"Score": "{:.4f}"}))


In [ ]:
# 16. Detailed classification report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Spam", "Spam"],
        zero_division=0
    )
)


In [ ]:
# 17. Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Spam", "Spam"],
    yticklabels=["Not Spam", "Spam"]
)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix")
plt.show()

print("Confusion matrix:")
print(cm)


In [ ]:
# 18. ROC Curve

fpr, tpr, thresholds = roc_curve(y_test, y_probability)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
# 19. Model coefficients / feature importance for Logistic Regression

classifier = model.named_steps["classifier"]

coef_df = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": classifier.coef_[0]
})

coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("Absolute_Coefficient", ascending=False)

display(coef_df)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=coef_df,
    x="Coefficient",
    y="Feature"
)
plt.title("Logistic Regression Feature Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.show()


## 20. Interpretation

For Logistic Regression:
- A **positive coefficient** means higher values of that feature increase the model's tendency toward the Spam class.
- A **negative coefficient** means higher values of that feature decrease the model's tendency toward the Spam class.
- Coefficients should be interpreted after feature scaling and in the context of the dataset.

This gives an interpretable explanation that can be discussed during the viva.


In [ ]:
# 21. Save the trained model

artifact = {
    "model": model,
    "features": feature_columns,
    "target": target,
    "class_names": {0: "Not Spam", 1: "Spam"}
}

joblib.dump(artifact, MODEL_PATH)

print(f"Model saved successfully to: {MODEL_PATH}")


In [ ]:
# 22. Test the saved model on one sample

loaded_artifact = joblib.load(MODEL_PATH)
loaded_model = loaded_artifact["model"]

sample = X_test.iloc[[0]]
prediction = int(loaded_model.predict(sample)[0])
probability = float(loaded_model.predict_proba(sample)[0, 1])

print("Sample input:")
display(sample)

print("Prediction:", "Spam" if prediction == 1 else "Not Spam")
print(f"Spam probability: {probability:.2%}")


## 23. Conclusion

The project implements a complete machine-learning workflow for email spam classification: data loading, data-quality checking, exploratory analysis, feature preparation, train/test splitting, Logistic Regression training, evaluation, interpretation, model persistence, and deployment preparation.

### Limitations
1. The dataset contains engineered numerical features rather than raw email text.
2. Therefore, text-based NLP features such as TF-IDF and threat-keyword counts cannot be honestly claimed from this dataset.
3. The model is intended as an academic prototype and should not be treated as a production email-security system.
4. Performance depends on the quality and representativeness of the supplied dataset.

### Future Scope
- Add raw email text and TF-IDF/NLP features.
- Compare Logistic Regression with Decision Tree, Random Forest, and other classifiers.
- Add cross-validation and hyperparameter tuning.
- Use a larger and more diverse dataset.
- Add explainability and threshold tuning.
- Deploy the model behind authentication and monitoring in a production environment.
